[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/NEURAL_BIZ/blob/main/Bishop_Ch_05/NB_bishop_ch05_credit_scoring.ipynb)


# Chapter 5 — Classification: Credit Scoring

This notebook applies the classification techniques from Bishop Chapter 5 to a
**credit-scoring** problem: predicting whether a loan applicant will default.

We compare Logistic Regression and Gaussian Naive Bayes, examine ROC curves,
optimise the decision threshold under an asymmetric cost function, and discuss
regulatory requirements under the EU AI Act.

---
## 1 Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    classification_report,
    roc_curve,
    roc_auc_score,
    confusion_matrix,
)
from sklearn.preprocessing import StandardScaler

# ── Course colours ──────────────────────────────────────────────────
BLUE    = "#1f4e79"
CRIMSON = "#c00000"
GREEN   = "#2e7d32"

# ── Reproducibility ─────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

print("Setup complete.")

---
## 2 Generate Synthetic Credit Data

We use `make_classification` with an imbalanced class distribution
(≈ 5 % default rate) to mimic a realistic credit portfolio.

In [ ]:
FEATURE_NAMES = [
    "Income",
    "Debt_to_Income",
    "Credit_History_Yrs",
    "Num_Open_Accounts",
    "Loan_Amount",
    "Employment_Yrs",
    "Credit_Utilisation",
    "Num_Late_Payments",
]

X, y = make_classification(
    n_samples=5000,
    n_features=8,
    n_informative=6,
    n_redundant=1,
    n_clusters_per_class=2,
    weights=[0.95, 0.05],
    flip_y=0.01,
    random_state=SEED,
)

df = pd.DataFrame(X, columns=FEATURE_NAMES)
df["Default"] = y

print(f"Dataset shape : {df.shape}")
print(f"Default rate  : {y.mean():.2%}")
print()
df.head()

---
## 3 Train / Test Split

We use a **stratified** 70 / 30 split so the default rate is preserved in both
sets.

> **Production note:** In a real credit-scoring pipeline you would typically
> use a *chronological* (time-based) split — training on older applications and
> testing on newer ones — because future economic conditions differ from past
> ones.  A random split overstates performance by allowing information leakage
> across time.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=SEED
)

# Standardise features (important for Logistic Regression coefficients)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"Train: {X_train_s.shape[0]:,} samples  |  Default rate: {y_train.mean():.2%}")
print(f"Test : {X_test_s.shape[0]:,} samples  |  Default rate: {y_test.mean():.2%}")

---
## 4 Fit Models

We compare two classifiers from Chapter 5:

| Model | Bishop Section | Key Assumption |
|---|---|---|
| **Logistic Regression** | §5.2 — discriminative | Models $p(C_k \mid \mathbf{x})$ directly via a sigmoid / softmax |
| **Gaussian Naive Bayes** | §5.1 — generative | Models $p(\mathbf{x} \mid C_k)$ as multivariate Gaussian with diagonal covariance |

In [ ]:
# ── Logistic Regression ─────────────────────────────────────────────
lr = LogisticRegression(max_iter=1000, random_state=SEED)
lr.fit(X_train_s, y_train)
y_pred_lr = lr.predict(X_test_s)

print("=" * 55)
print("LOGISTIC REGRESSION — Classification Report")
print("=" * 55)
print(classification_report(y_test, y_pred_lr, target_names=["No Default", "Default"]))

In [ ]:
# ── Gaussian Naive Bayes ────────────────────────────────────────────
gnb = GaussianNB()
gnb.fit(X_train_s, y_train)
y_pred_gnb = gnb.predict(X_test_s)

print("=" * 55)
print("GAUSSIAN NAIVE BAYES — Classification Report")
print("=" * 55)
print(classification_report(y_test, y_pred_gnb, target_names=["No Default", "Default"]))

---
## 5 ROC Curves

The ROC curve plots the **True Positive Rate** (recall for defaults) against
the **False Positive Rate** as the decision threshold varies from 0 to 1.
A model whose curve hugs the top-left corner is better at separating the two
classes.  The **AUC** (Area Under the Curve) summarises this in a single number.

In [ ]:
# Predicted probabilities for class 1 (Default)
prob_lr  = lr.predict_proba(X_test_s)[:, 1]
prob_gnb = gnb.predict_proba(X_test_s)[:, 1]

fpr_lr,  tpr_lr,  _  = roc_curve(y_test, prob_lr)
fpr_gnb, tpr_gnb, _  = roc_curve(y_test, prob_gnb)

auc_lr  = roc_auc_score(y_test, prob_lr)
auc_gnb = roc_auc_score(y_test, prob_gnb)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr_lr,  tpr_lr,  color=BLUE,    lw=2, label=f"Logistic Reg  (AUC = {auc_lr:.3f})")
ax.plot(fpr_gnb, tpr_gnb, color=CRIMSON, lw=2, label=f"Naive Bayes   (AUC = {auc_gnb:.3f})")
ax.plot([0, 1], [0, 1], ls="--", color="grey", lw=1, label="Random")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves — Credit Scoring")
ax.legend(loc="lower right", frameon=False)
plt.tight_layout()
plt.show()

---
## 6 Cost-Sensitive Threshold Optimisation

The default 0.5 threshold treats false positives and false negatives equally.
In credit scoring the costs are highly **asymmetric**:

| Outcome | Financial Impact |
|---|---|
| **False Negative** (approve a defaulter) | Lose the outstanding loan amount |
| **False Positive** (reject a good borrower) | Forgo the profit margin on that loan |

We sweep thresholds from 0 to 1, compute the total cost on the test set, and
pick the threshold that minimises it.

In [ ]:
loan_amount   = 10_000   # cost incurred per undetected default (FN)
profit_margin =    500   # revenue lost per wrongly rejected good loan (FP)

thresholds = np.linspace(0.01, 0.99, 300)
total_costs = []

for t in thresholds:
    y_pred_t = (prob_lr >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_t).ravel()
    cost = fn * loan_amount + fp * profit_margin
    total_costs.append(cost)

total_costs = np.array(total_costs)
best_idx = np.argmin(total_costs)
best_threshold = thresholds[best_idx]
best_cost = total_costs[best_idx]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(thresholds, total_costs / 1e6, color=BLUE, lw=2)
ax.axvline(best_threshold, color=CRIMSON, ls="--", lw=1.5,
           label=f"Optimal threshold = {best_threshold:.3f}")
ax.axvline(0.5, color="grey", ls=":", lw=1, label="Default threshold = 0.50")
ax.set_xlabel("Decision Threshold")
ax.set_ylabel("Total Cost ($ millions)")
ax.set_title("Cost-Sensitive Threshold Optimisation (Logistic Regression)")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

print(f"Optimal threshold : {best_threshold:.3f}")
print(f"Minimum total cost: ${best_cost:,.0f}")
print()
print("Classification report at the optimal threshold:")
y_pred_opt = (prob_lr >= best_threshold).astype(int)
print(classification_report(y_test, y_pred_opt, target_names=["No Default", "Default"]))

---
## 7 Feature Importance

Because we standardised the inputs, the **magnitude** of each logistic-regression
coefficient tells us how strongly that feature shifts the log-odds of default,
and the **sign** tells us the direction.

In [ ]:
coefs = lr.coef_[0]
sorted_idx = np.argsort(np.abs(coefs))

fig, ax = plt.subplots(figsize=(7, 4.5))
colors = [CRIMSON if c > 0 else BLUE for c in coefs[sorted_idx]]
ax.barh(
    [FEATURE_NAMES[i] for i in sorted_idx],
    coefs[sorted_idx],
    color=colors,
    edgecolor="white",
    linewidth=0.5,
)
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("Logistic Regression Coefficient (standardised)")
ax.set_title("Feature Importance — Credit Scoring")

# Legend
from matplotlib.patches import Patch
ax.legend(
    handles=[
        Patch(facecolor=CRIMSON, label="Increases default risk"),
        Patch(facecolor=BLUE,    label="Decreases default risk"),
    ],
    loc="lower right",
    frameon=False,
)
plt.tight_layout()
plt.show()

---
## 8 Regulatory Context — EU AI Act

Credit scoring is classified as a **high-risk AI system** under the
[EU AI Act (2024)](https://artificialintelligenceact.eu/), Annex III §5(b).

### Key obligations for high-risk credit-scoring models

| Requirement | What it means in practice |
|---|---|
| **Transparency (Art. 13)** | The institution must be able to explain *why* an applicant was rejected in terms a non-expert can understand. |
| **Human oversight (Art. 14)** | A qualified human must be able to override the model's decision. Fully automated rejections without review are restricted. |
| **Data governance (Art. 10)** | Training data must be checked for biases related to protected attributes (gender, ethnicity, age). |
| **Risk management (Art. 9)** | The model must be continuously monitored for performance drift, and a documented risk-management system must be in place. |
| **Record-keeping (Art. 12)** | Logs of individual decisions (inputs, outputs, threshold used) must be retained for audit. |

### Implications for model choice

- **Logistic Regression** is often preferred in regulated settings because its
  coefficients provide a straightforward explanation: each feature has a signed
  weight whose magnitude shows its importance.
- **Naive Bayes** also has an interpretable generative structure (per-class
  likelihoods), though it is used less frequently in production credit scoring.
- More powerful but opaque models (gradient-boosted trees, neural networks)
  can be deployed, but they require additional explainability tooling
  (e.g., SHAP values) to satisfy transparency obligations.

---
## 9 Key Takeaways

1. **Class imbalance matters.** With only ~5 % defaults, accuracy alone is
   misleading — a model that always predicts "no default" achieves 95 %
   accuracy but catches zero actual defaults.

2. **ROC-AUC separates discrimination from calibration.** It tells us how
   well the model *ranks* applicants by risk, independent of the threshold.

3. **The decision threshold is a business decision, not a statistical one.**
   The optimal threshold depends on the ratio of default cost to
   missed-profit cost — and that ratio changes with the economic cycle.

4. **Generative vs. discriminative.** Logistic Regression (discriminative)
   typically outperforms Naive Bayes (generative) when training data is
   plentiful, consistent with Bishop §5.2's discussion.

5. **Explainability is a legal requirement.** Under the EU AI Act,
   credit-scoring models must be transparent, auditable, and subject to
   human oversight.